# Sentiment Analysis

In [1]:
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

## Naive Bayes Classifier for Sentiment Analysis

In [4]:
# Training data
docs = [
    ("I love this movie", "positive"),
    ("This film is amazing", "positive"),
    ("I hate this movie", "negative"),
    ("This film is terrible", "negative"),
]

# Step 1: Extract vocabulary
vocab = sorted(set(word.lower() for doc, _ in docs for word in doc.split()))
print("Vocabulary:", vocab)

# Step 2: Compute P(c)
classes = set(label for _, label in docs)
P_c = {}
for c in classes:
    P_c[c] = sum(1 for _, label in docs if label == c) / len(docs)
print("\nP(c):", P_c)

# Step 3: Compute P(w | c) with Laplace smoothing
P_w_given_c = {}

for c in classes:
    # Concatenate all words from docs of class c
    all_words = [word.lower() for doc, label in docs if label == c for word in doc.split()]
    n = len(all_words)
    P_w_given_c[c] = {}
    
    for w in vocab:
        ni = all_words.count(w)
        # Laplace smoothing
        P_w_given_c[c][w] = (ni + 1) / (n + len(vocab))

print("\nP(w | c):")
for c in P_w_given_c:
    print(f"  {c}: {P_w_given_c[c]}")

# Step 4: Prediction function
def predict(doc):
    words = doc.lower().split()
    scores = {}
    for c in classes:
        # log probabilities prevent underflow
        log_prob = np.log(P_c[c])
        for w in words:
            if w in vocab:
                log_prob += np.log(P_w_given_c[c][w])
        scores[c] = log_prob
    return max(scores, key=scores.get), scores

def predict_with_probs(doc):
    words = doc.lower().split()
    probs = {}
    for c in classes:
        log_prob = np.log(P_c[c])
        for w in words:
            if w in vocab:
                log_prob += np.log(P_w_given_c[c][w])
        probs[c] = np.exp(log_prob)  # back to normal space
    
    # normalize to sum to 1
    total = sum(probs.values())
    for c in probs:
        probs[c] /= total
    return max(probs, key=probs.get), probs

test_doc = "I love this film"
pred_label, scores = predict(test_doc)
print(f"\nDocument: '{test_doc}'")
print("Scores:", scores)
print("Normalized probs", predict_with_probs(test_doc)[1])
print("Predicted label:", pred_label)


Vocabulary: ['amazing', 'film', 'hate', 'i', 'is', 'love', 'movie', 'terrible', 'this']

P(c): {'negative': 0.5, 'positive': 0.5}

P(w | c):
  negative: {'amazing': 0.058823529411764705, 'film': 0.11764705882352941, 'hate': 0.11764705882352941, 'i': 0.11764705882352941, 'is': 0.11764705882352941, 'love': 0.058823529411764705, 'movie': 0.11764705882352941, 'terrible': 0.11764705882352941, 'this': 0.17647058823529413}
  positive: {'amazing': 0.11764705882352941, 'film': 0.11764705882352941, 'hate': 0.058823529411764705, 'i': 0.11764705882352941, 'is': 0.11764705882352941, 'love': 0.11764705882352941, 'movie': 0.11764705882352941, 'terrible': 0.058823529411764705, 'this': 0.17647058823529413}

Document: 'I love this film'
Scores: {'negative': -9.54109390699681, 'positive': -8.847946726436863}
Normalized probs {'negative': 0.3333333333333329, 'positive': 0.6666666666666671}
Predicted label: positive


## Using Scikit-learn for Comparison

In [3]:
# Prepare data
texts = [doc for doc, _ in docs]
labels = [label for _, label in docs]

# Convert text to bag-of-words
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

# Train Naive Bayes
clf = MultinomialNB()
clf.fit(X, labels)

# Predict
test_vec = vectorizer.transform([test_doc])
sk_pred = clf.predict(test_vec)[0]
sk_probs = clf.predict_proba(test_vec)

print("\n--- Scikit-learn Comparison ---")
print("Predicted label:", sk_pred)
print("Class probabilities:", dict(zip(clf.classes_, sk_probs[0])))



--- Scikit-learn Comparison ---
Predicted label: positive
Class probabilities: {'negative': 0.3333333333333332, 'positive': 0.6666666666666665}
